## Complete Batch

In [0]:
%run "../00_common/01_environment_config"

In [0]:
# dbutils.widgets.text("p_batch_id", "")
# v_batch_id = dbutils.widgets.get("p_batch_id").strip()

# if not v_batch_id:
#     raise Exception("batch_id is missing")

# print(f"Batch ID: {v_batch_id}")

In [0]:

# MAGIC %run ../00_common/01_environment_config

dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id").strip()

if not v_batch_id:
    raise Exception("batch_id is missing")

from delta.tables import DeltaTable
from pyspark.sql import functions as F

control_table_path = (
    f"{s3_root_path}/Control/Kalashri_increment/batch_control"
)

delta_table = DeltaTable.forPath(spark, control_table_path)

source_df = (
    spark.createDataFrame([(v_batch_id,)], ["batch_id"])
    .withColumn("status", F.lit("completed"))
    .withColumn("updated_timestamp", F.current_timestamp())
)

(
    delta_table.alias("t")
    .merge(
        source_df.alias("s"),
        "t.batch_id = s.batch_id AND t.status = 'in_progress'"
    )
    .whenMatchedUpdate(set={
        "status": "s.status",
        "updated_timestamp": "s.updated_timestamp"
    })
    .execute()
)

print(f"Batch {v_batch_id} marked as completed.")

display(
    spark.read.format("delta")
    .load(control_table_path)
    .filter(F.col("batch_id") == v_batch_id)
)


In [0]:
# from delta.tables import DeltaTable
# from pyspark.sql import functions as F

# # Control table path
# control_table_path = "s3://bizf1project123/Control/Kalashri_increment/batch_control"

# # Load the existing Delta table
# delta_table = DeltaTable.forPath(spark, control_table_path)

# # Prepare batch status update
# source_df = (
#     spark.createDataFrame([(v_batch_id,)], ["batch_id"])
#     .withColumn("status", F.lit("completed"))
#     .withColumn("updated_timestamp", F.current_timestamp())
# )

# # Update the batch status
# (
#     delta_table.alias("t")
#     .merge(
#         source_df.alias("s"),
#         "t.batch_id = s.batch_id AND t.status = 'in_progress'"
#     )
#     .whenMatchedUpdate(set={
#         "status": "s.status",
#         "updated_timestamp": "s.updated_timestamp"
#     })
#     .execute()
# )

# print(f"Batch {v_batch_id} marked as completed.")

In [0]:
batch_df = (
    spark.read.format("delta")
    .load(control_table_path)
    .filter(F.col("batch_id") == v_batch_id)
)

display(batch_df)